# Анализ воронки конверсии и выручки интернет-магазина

## Цель проекта

Проанализировать события пользователей интернет-магазина и понять:

- на каких этапах воронки теряются пользователи;
- какие источники трафика дают более качественную аудиторию;
- как отличается поведение пользователей на разных устройствах;
- какие категории товаров приносят наибольшую выручку.

## Что считается результатом анализа

В конце исследования должны быть получены понятные выводы и рекомендации: где находится главная проблема воронки, какие сегменты выглядят сильнее, а какие требуют дополнительной проверки.

## 1. Загрузка библиотек и данных

В проекте используются `pandas` для анализа данных, `sqlite3` для SQL-запросов и `matplotlib` для простых визуализаций.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
# Ищем файл с данными в нескольких возможных местах,
# чтобы ноутбук запускался и из папки notebooks, и из корня проекта.
possible_paths = [
    Path('../data/events.csv'),
    Path('data/events.csv'),
    Path('events.csv')
]

for path in possible_paths:
    if path.exists():
        data_path = path
        break
else:
    raise FileNotFoundError('Файл events.csv не найден. Проверь путь к данным.')

df = pd.read_csv(data_path)

print(f'Файл загружен: {data_path}')
print(f'Размер таблицы: {df.shape[0]} строк и {df.shape[1]} колонок')
df.head()

## 2. Первичный обзор данных

Сначала посмотрим структуру таблицы: типы данных, пропуски, дубликаты и базовые значения в ключевых колонках.

In [ ]:
df.info()

In [ ]:
missing = (
    df.isna()
      .sum()
      .reset_index()
      .rename(columns={'index': 'column', 0: 'missing_count'})
)
missing['missing_share'] = (missing['missing_count'] / len(df) * 100).round(2)
missing.sort_values('missing_count', ascending=False)

In [ ]:
print('Количество полных дубликатов:', df.duplicated().sum())

for column in ['event_type', 'device_type', 'traffic_source', 'category', 'region']:
    if column in df.columns:
        print(f'
{column}:')
        display(df[column].value_counts(dropna=False).head(10))

### Промежуточный вывод по качеству данных

В данных есть пропуски в полях, связанных с товаром и заказом. Это ожидаемо: не каждое событие связано с конкретным товаром или покупкой. Например, визит на сайт может не иметь `product_id`, `category` и `order_id`.

Полные дубликаты стоит проверить отдельно. Если они есть, их можно удалить, потому что одинаковые строки могут исказить количество событий и выручку.

## 3. Подготовка данных

Приведём даты к правильному типу и удалим полные дубликаты, если они есть.

In [ ]:
df_clean = df.copy()

# Приводим даты к формату datetime.
df_clean['event_ts'] = pd.to_datetime(df_clean['event_ts'])
df_clean['event_date'] = pd.to_datetime(df_clean['event_date'])

# Удаляем полные дубликаты, если они есть.
df_clean = df_clean.drop_duplicates()

print(f'Размер таблицы после подготовки: {df_clean.shape[0]} строк и {df_clean.shape[1]} колонок')
print('Период данных:', df_clean['event_date'].min().date(), '—', df_clean['event_date'].max().date())

## 4. Ключевые метрики проекта

Посчитаем основные показатели интернет-магазина:

- количество уникальных посетителей;
- количество заказов;
- выручку;
- средний чек;
- конверсию в покупку.

In [ ]:
visitors = df_clean['user_id'].nunique()
orders = df_clean.loc[df_clean['order_id'].notna(), 'order_id'].nunique()
revenue = df_clean['revenue'].sum()
avg_order_value = revenue / orders if orders else 0
buyers = df_clean.loc[df_clean['event_type'] == 'purchase', 'user_id'].nunique()
purchase_conversion = buyers / visitors * 100 if visitors else 0

metrics = pd.DataFrame({
    'metric': [
        'Уникальные посетители',
        'Количество заказов',
        'Выручка',
        'Средний чек',
        'Покупатели',
        'Конверсия в покупку, %'
    ],
    'value': [
        visitors,
        orders,
        revenue,
        avg_order_value,
        buyers,
        purchase_conversion
    ]
})

metrics

## 5. Анализ событий

Посмотрим, какие события встречаются чаще всего. Это помогает понять, как пользователи двигаются по сайту и где начинается потеря аудитории.

In [ ]:
event_counts = (
    df_clean['event_type']
    .value_counts()
    .rename_axis('event_type')
    .reset_index(name='events_count')
)

event_counts

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(event_counts['event_type'], event_counts['events_count'])
plt.title('Количество событий по типам')
plt.xlabel('Тип события')
plt.ylabel('Количество событий')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Воронка конверсии через SQL

Для портфолио важно показать не только `pandas`, но и SQL. Поэтому рассчитаем воронку по уникальным пользователям через SQLite.

Этапы воронки:

1. `visit` — пользователь посетил сайт;
2. `view_item` — посмотрел товар;
3. `add_to_cart` — добавил товар в корзину;
4. `checkout` — перешёл к оформлению;
5. `purchase` — совершил покупку.

In [ ]:
conn = sqlite3.connect(':memory:')
df_clean.to_sql('events', conn, index=False, if_exists='replace')

In [ ]:
funnel_query = """
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_type = 'visit' THEN user_id END) AS visitors,
        COUNT(DISTINCT CASE WHEN event_type = 'view_item' THEN user_id END) AS viewers,
        COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS cart_adders,
        COUNT(DISTINCT CASE WHEN event_type = 'checkout' THEN user_id END) AS checkouts,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
    FROM events
)
SELECT
    visitors,
    viewers,
    cart_adders,
    checkouts,
    buyers,
    ROUND(buyers * 100.0 / visitors, 2) AS purchase_conversion
FROM funnel;
"""

funnel_sql = pd.read_sql(funnel_query, conn)
funnel_sql

In [ ]:
funnel = pd.DataFrame({
    'stage': ['Визит', 'Просмотр товара', 'Добавление в корзину', 'Оформление', 'Покупка'],
    'users': [
        funnel_sql.loc[0, 'visitors'],
        funnel_sql.loc[0, 'viewers'],
        funnel_sql.loc[0, 'cart_adders'],
        funnel_sql.loc[0, 'checkouts'],
        funnel_sql.loc[0, 'buyers']
    ]
})

funnel['conversion_from_start'] = (funnel['users'] / funnel.loc[0, 'users'] * 100).round(2)
funnel['conversion_from_previous_stage'] = (funnel['users'] / funnel['users'].shift(1) * 100).round(2)
funnel['drop_from_previous_stage'] = (100 - funnel['conversion_from_previous_stage']).round(2)

funnel

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(funnel['stage'], funnel['users'])
plt.title('Воронка по уникальным пользователям')
plt.xlabel('Этап воронки')
plt.ylabel('Количество пользователей')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

### Вывод по воронке

Главная точка потерь определяется по колонке `drop_from_previous_stage`. Чем больше потеря между двумя соседними этапами, тем важнее проверить этот участок пользовательского пути.

Особенно важно смотреть переход от просмотра товара к добавлению в корзину: если пользователь уже заинтересовался товаром, но не добавил его в корзину, проблема может быть в цене, карточке товара, условиях доставки, наличии товара или качестве интерфейса.

## 7. Источники трафика

Сравним источники трафика по трем показателям:

- сколько пользователей пришло из источника;
- какая была конверсия в покупку;
- какую выручку принёс источник.

In [ ]:
purchases = df_clean[df_clean['event_type'] == 'purchase']

source_users = df_clean.groupby('traffic_source')['user_id'].nunique().reset_index(name='users')
source_buyers = purchases.groupby('traffic_source')['user_id'].nunique().reset_index(name='buyers')
source_orders = df_clean.groupby('traffic_source')['order_id'].nunique().reset_index(name='orders')
source_revenue = df_clean.groupby('traffic_source')['revenue'].sum().reset_index(name='revenue')

source_analysis = (
    source_users
    .merge(source_buyers, on='traffic_source', how='left')
    .merge(source_orders, on='traffic_source', how='left')
    .merge(source_revenue, on='traffic_source', how='left')
)

source_analysis[['buyers', 'orders', 'revenue']] = source_analysis[['buyers', 'orders', 'revenue']].fillna(0)
source_analysis['conversion_to_purchase'] = (source_analysis['buyers'] / source_analysis['users'] * 100).round(2)
source_analysis['avg_order_value'] = (source_analysis['revenue'] / source_analysis['orders']).round(2)
source_analysis = source_analysis.sort_values('revenue', ascending=False)

source_analysis

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(source_analysis['traffic_source'], source_analysis['conversion_to_purchase'])
plt.title('Конверсия в покупку по источникам трафика')
plt.xlabel('Источник трафика')
plt.ylabel('Конверсия в покупку, %')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(source_analysis['traffic_source'], source_analysis['revenue'])
plt.title('Выручка по источникам трафика')
plt.xlabel('Источник трафика')
plt.ylabel('Выручка')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Вывод по источникам

Источник трафика нужно оценивать не только по количеству пользователей. Важнее смотреть связку: `users`, `conversion_to_purchase`, `revenue` и `avg_order_value`.

Источник может приводить много пользователей, но давать слабую конверсию. И наоборот: источник может быть небольшим по объёму, но качественным по покупкам и среднему чеку.

## 8. Анализ по типам устройств

Проверим, отличаются ли пользователи с разных устройств по конверсии и выручке.

In [ ]:
device_users = df_clean.groupby('device_type')['user_id'].nunique().reset_index(name='users')
device_buyers = purchases.groupby('device_type')['user_id'].nunique().reset_index(name='buyers')
device_orders = df_clean.groupby('device_type')['order_id'].nunique().reset_index(name='orders')
device_revenue = df_clean.groupby('device_type')['revenue'].sum().reset_index(name='revenue')

device_analysis = (
    device_users
    .merge(device_buyers, on='device_type', how='left')
    .merge(device_orders, on='device_type', how='left')
    .merge(device_revenue, on='device_type', how='left')
)

device_analysis[['buyers', 'orders', 'revenue']] = device_analysis[['buyers', 'orders', 'revenue']].fillna(0)
device_analysis['conversion_to_purchase'] = (device_analysis['buyers'] / device_analysis['users'] * 100).round(2)
device_analysis['avg_order_value'] = (device_analysis['revenue'] / device_analysis['orders']).round(2)
device_analysis = device_analysis.sort_values('conversion_to_purchase', ascending=False)

device_analysis

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(device_analysis['device_type'], device_analysis['conversion_to_purchase'])
plt.title('Конверсия в покупку по типам устройств')
plt.xlabel('Тип устройства')
plt.ylabel('Конверсия в покупку, %')
plt.tight_layout()
plt.show()

### Вывод по устройствам

Если один тип устройства заметно проигрывает по конверсии, это повод отдельно проверить пользовательский путь: карточку товара, корзину, оформление заказа, скорость загрузки и удобство интерфейса.

## 9. Категории товаров

Посмотрим, какие категории дают основной вклад в выручку и количество заказов.

In [ ]:
category_analysis = (
    df_clean[df_clean['category'].notna()]
    .groupby('category')
    .agg(
        events=('event_id', 'count'),
        users=('user_id', 'nunique'),
        orders=('order_id', 'nunique'),
        revenue=('revenue', 'sum')
    )
    .reset_index()
)

category_analysis['avg_order_value'] = (category_analysis['revenue'] / category_analysis['orders']).round(2)
category_analysis = category_analysis.sort_values('revenue', ascending=False)

category_analysis

In [ ]:
top_categories = category_analysis.head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_categories['category'], top_categories['revenue'])
plt.title('Топ категорий по выручке')
plt.xlabel('Категория')
plt.ylabel('Выручка')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Вывод по категориям

Категории с высокой выручкой стоит анализировать отдельно: они сильнее всего влияют на общий результат магазина. Для таких категорий особенно важны наличие товара, качество карточек, цена, доставка и видимость в каталоге.

## 10. Новые и старые пользователи

Проверим, отличаются ли новые пользователи от уже знакомых магазину пользователей.

In [ ]:
new_users = df_clean.groupby('is_new_user')['user_id'].nunique().reset_index(name='users')
new_buyers = purchases.groupby('is_new_user')['user_id'].nunique().reset_index(name='buyers')
new_orders = df_clean.groupby('is_new_user')['order_id'].nunique().reset_index(name='orders')
new_revenue = df_clean.groupby('is_new_user')['revenue'].sum().reset_index(name='revenue')

new_user_analysis = (
    new_users
    .merge(new_buyers, on='is_new_user', how='left')
    .merge(new_orders, on='is_new_user', how='left')
    .merge(new_revenue, on='is_new_user', how='left')
)

new_user_analysis[['buyers', 'orders', 'revenue']] = new_user_analysis[['buyers', 'orders', 'revenue']].fillna(0)
new_user_analysis['user_type'] = new_user_analysis['is_new_user'].map({0: 'Старые пользователи', 1: 'Новые пользователи'})
new_user_analysis['conversion_to_purchase'] = (new_user_analysis['buyers'] / new_user_analysis['users'] * 100).round(2)
new_user_analysis['avg_order_value'] = (new_user_analysis['revenue'] / new_user_analysis['orders']).round(2)

new_user_analysis[['user_type', 'users', 'buyers', 'orders', 'revenue', 'conversion_to_purchase', 'avg_order_value']]

## 11. Итоговые выводы

По результатам анализа можно выделить несколько ключевых наблюдений:

1. Воронка показывает, на каком этапе пользователи теряются сильнее всего. Главный участок для проверки — переход между соседними этапами с максимальной потерей.
2. Источники трафика отличаются не только объёмом пользователей, но и качеством: конверсией, выручкой и средним чеком.
3. Разница между устройствами может указывать на проблемы в интерфейсе или процессе оформления заказа.
4. Основную выручку дают отдельные категории товаров, поэтому их нужно анализировать глубже.

## Рекомендации

1. Подробно проверить этап с максимальной потерей пользователей в воронке.
2. Отдельно изучить карточки товаров и причины, по которым пользователи не добавляют товар в корзину.
3. Проверить мобильный сценарий покупки, если мобильные устройства показывают более слабую конверсию.
4. Перераспределять маркетинговое внимание не только по объёму трафика, но и по качеству источника.
5. Для категорий с наибольшей выручкой контролировать наличие товаров, цены, доставку и качество карточек.

